In [1]:
import os
import pytest
import ee
from typing import Literal

In [2]:
ee.Authenticate()
ee.Initialize()
from EeImageCollections.MultiOrbitCollection import MultiOrbitCollection
from EeImageCollections.SingleOrbitCollection import SingleOrbitCollection
from EeImageCollections.S1_GRD.S1GRD_MultiOrbitCollection import S1GRD_MultiOrbitCollection
from EeImageCollections.S1_GRD.S1GRD_SingleOrbitCollection import S1GRD_SingleOrbitCollection


In [3]:
assets = ee.FeatureCollection('projects/ee-zachariasguislain/assets/LeuvenFields')
roi = assets.bounds()
db_collection = S1GRD_MultiOrbitCollection.of(roi, '2024-01-01', '2024-12-31', 'db')
linear_collection=  S1GRD_MultiOrbitCollection.of(roi, '2024-01-01', '2024-12-31', 'linear')


In [4]:
db_37 = S1GRD_SingleOrbitCollection.of(db_collection, 37)

In [5]:
print(db_37.get_bands())
print(db_37.get_relative_orbits())
print(db_37.get_relative_orbit())
print(db_37.get_pass_type())
print(db_37.get_time_period())
print(db_37.get_start_year())
print(db_37.get_relative_orbit_info())
print(db_37.get_collection_name())
print(db_37.get_collection_id())
print(db_37.get_footprint_size())
print(db_37.get_mother_collection_id())
print(db_37.get_units())
print(db_37 == db_37.of(db_collection, 37))


{'VH', 'VV', 'angle'}
{37}
37
DESCENDING
('2024-01-01', '2024-12-31')
2024
{37: 'DESCENDING'}
COPERNICUS/S1_GRD
de970ba12c48a13748ed1bff734332971ff067ae5428c4efcb09cb23c81fe89b
45000000
3a1834b72cda3452c5e4c5b41f178ed3fe8ef5fe068821267a09c9ac455b6eee
db
True


In [6]:
db_37.add_db_cross_polarization()
print(db_37.get_bands())
cp_band = db_37._collection.first().select('CP')
stats = cp_band.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=cp_band.geometry(),  # or your own region/AOI
    scale=30,                   # match the image's native resolution
    maxPixels=1e13
)

print(stats.getInfo())

{'VH', 'VV', 'CP', 'angle'}
{'CP_max': 17.764318939772004, 'CP_min': -45.07359762031699}


In [7]:
db_collection = S1GRD_MultiOrbitCollection.of(roi, '2024-01-01', '2024-12-31', 'db')
db_37 = S1GRD_SingleOrbitCollection.of(db_collection, 37)
db_37.add_linear_cross_polarization()
print(db_37.get_bands())
cp_band = db_37._collection.first().select('CP')
stats = cp_band.reduceRegion(
    reducer=ee.Reducer.minMax().combine(ee.Reducer.mean(), sharedInputs=True),
    geometry=cp_band.geometry(),  # or your own region/AOI
    scale=30,                   # match the image's native resolution
    maxPixels=1e13
)

print(stats.getInfo())

{'VH', 'VV', 'CP', 'angle'}
{'CP_max': 59.762931716229126, 'CP_mean': 0.2675846329905471, 'CP_min': 3.109139712515336e-05}


In [8]:
linear_37 = S1GRD_SingleOrbitCollection.of(linear_collection, 37)
linear_37.add_db_cross_polarization()
print(linear_37.get_bands())
cp_band = linear_37._collection.first().select('CP')
stats = cp_band.reduceRegion(
    reducer=ee.Reducer.minMax().combine(ee.Reducer.mean(), sharedInputs=True),
    geometry=cp_band.geometry(),  # or your own region/AOI
    scale=30,                   # match the image's native resolution
    maxPixels=1e13
)

print(stats.getInfo())

{'VH', 'VV', 'CP', 'angle'}
{'CP_max': 17.764318939772004, 'CP_mean': -6.58621921444527, 'CP_min': -45.07359762031698}


In [9]:
linear_37 = S1GRD_SingleOrbitCollection.of(linear_collection, 37)
linear_37.add_linear_cross_polarization()
print(linear_37.get_bands())
cp_band = linear_37._collection.first().select('CP')
stats = cp_band.reduceRegion(
    reducer=ee.Reducer.minMax().combine(ee.Reducer.mean(), sharedInputs=True),
    geometry=cp_band.geometry(),  # or your own region/AOI
    scale=30,                   # match the image's native resolution
    maxPixels=1e13
)

print(stats.getInfo())

{'VH', 'VV', 'CP', 'angle'}
{'CP_max': 59.76293171622913, 'CP_mean': 0.2675846329905471, 'CP_min': 3.109139712515339e-05}


In [10]:
import geemap
my_map = geemap.Map()
my_map.centerObject(roi)
my_map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

In [11]:
my_map.addLayer(db_37._collection.select('VV').first(), name='VV (db)')

In [12]:
linear_37.filter_border_noise()

vv_band = linear_37._collection.first().select('VV')
stats = vv_band.reduceRegion(
    reducer=ee.Reducer.min(),
    geometry=vv_band.geometry(),  # or your own region/AOI
    scale=30,                   # match the image's native resolution
    maxPixels=1e13
)

print(stats.getInfo()['VV'] >= 0.001)

angle_band = linear_37._collection.first().select('angle')
stats = angle_band.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=angle_band.geometry(),  # or your own region/AOI
    scale=30,                   # match the image's native resolution
    maxPixels=1e13
)

print(stats.getInfo()['angle_min'] > 28)
print(stats.getInfo()['angle_max'] < 47)

True
True
True


In [13]:
db_37.filter_border_noise()
my_map.addLayer(db_37.get_collection().first().select('VV'), name='VV (db) - border noise filtered')

vv_band = db_37._collection.first().select('VV')
stats = vv_band.reduceRegion(
    reducer=ee.Reducer.min(),
    geometry=vv_band.geometry(),  # or your own region/AOI
    scale=30,                   # match the image's native resolution
    maxPixels=1e13
)

print(stats.getInfo()['VV'] > -30)

angle_band = db_37._collection.first().select('angle')
stats = angle_band.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=angle_band.geometry(),  # or your own region/AOI
    scale=30,                   # match the image's native resolution
    maxPixels=1e13
)

print(stats.getInfo()['angle_min'] > 28)
print(stats.getInfo()['angle_max'] < 47)

True
True
True


In [14]:
db_37.add_azimuth_and_local_incidence_angle()
db_37.get_bands()

{'AZI', 'CP', 'LIA', 'VH', 'VV', 'angle'}

In [15]:
my_map.addLayer(db_37.get_collection().first().select('AZI'), name="azimuth")
my_map.addLayer(db_37.get_collection().first().select('LIA'), name="local incidence angle")

In [16]:
azi = db_37._collection.first().select('AZI')
stats = azi.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=azi.geometry(),  # or your own region/AOI
    scale=30,                   # match the image's native resolution
    maxPixels=1e13
)

stats

In [17]:
lia = db_37._collection.first().select('LIA')
stats = lia.reduceRegion(
    reducer=ee.Reducer.minMax().combine(ee.Reducer.mean(), sharedInputs=True).combine(ee.Reducer.stdDev(), sharedInputs = True),
    geometry=lia.geometry(),    # or your own region/AOI
    scale=30,                   # match the image's native resolution
    maxPixels=1e13
)

stats

In [18]:
db_37.get_name()

'S1_RO_37_Pass_DES_Year_2024'

In [19]:
db_37.get_bands_per_reducer()

{'mean_count_stdev': set(), 'mean_stdev': set(), 'mean': set()}

In [20]:
db_37.get_reducers()

{'mean_count_stdev': <ee.reducer.Reducer at 0x1f18f739d30>,
 'mean_stdev': <ee.reducer.Reducer at 0x1f18f73a990>,
 'mean': <ee.reducer.Reducer at 0x1f18f73a930>}

In [21]:

print(db_37.get_batch_size() == 500)
db_37.set_batch_size(20)
print(db_37.get_batch_size() == 20)

True
True


In [22]:
db_37.get_mother_collection_id() == db_collection.get_collection_id()

True

In [23]:
old_reducers = db_37.get_reducers()
db_37.add_reducer("min_max", ee.Reducer.minMax())
new_reducers = db_37.get_reducers()
print(old_reducers)
print(new_reducers)

{'mean_count_stdev': <ee.reducer.Reducer object at 0x000001F18F739D30>, 'mean_stdev': <ee.reducer.Reducer object at 0x000001F18F73A990>, 'mean': <ee.reducer.Reducer object at 0x000001F18F73A930>}
{'mean_count_stdev': <ee.reducer.Reducer object at 0x000001F18F739D30>, 'mean_stdev': <ee.reducer.Reducer object at 0x000001F18F73A990>, 'mean': <ee.reducer.Reducer object at 0x000001F18F73A930>, 'min_max': <ee.reducer.Reducer object at 0x000001F192C4AA20>}


In [24]:
db_37.allocate_band_to_reducer('mean_count_stdev', 'VV')
db_37.allocate_bands_to_reducer('mean_stdev', {'VH', 'LIA', 'CP'})
db_37.allocate_band_to_reducer('mean', 'AZI')

In [25]:
db_37.get_variables_after_reduction()

{'AZI_mean',
 'CP_mean',
 'CP_stdDev',
 'LIA_mean',
 'LIA_stdDev',
 'VH_mean',
 'VH_stdDev',
 'VV_count',
 'VV_mean',
 'VV_stdDev'}

In [26]:
db_37.reduce(assets, 'Name')